In [ ]:
from pathlib import Path
from typing import Literal, Optional

import joblib
import numpy as np
import pandas as pd

from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize


# ============================================================
# CONFIG
# ============================================================

PROCESSED_DIR = Path("../datasets/processed/PAN2011_300")
ARTIFACT_DIR = Path("../artifacts/tfidf")

SOURCE_CHUNKS_PATH = PROCESSED_DIR / "source_chunks_lsa_esa.parquet"
SUSPICIOUS_CHUNKS_PATH = PROCESSED_DIR / "suspicious_chunks_lsa_esa.parquet"
SOURCE_CANONICAL_CHUNKS_PATH = PROCESSED_DIR / "source_chunks.parquet"

SUSPICIOUS_DOC_ID = "part14__suspicious-document06510.txt"

OUTPUT_CANDIDATES_PATH = PROCESSED_DIR / "tfidf_candidates_suspicious_doc.parquet"
OUTPUT_TOP_DOCS_MAX_PATH = PROCESSED_DIR / "tfidf_top_source_documents_by_max_score.parquet"
OUTPUT_TOP_DOCS_MEAN_PATH = PROCESSED_DIR / "tfidf_top_source_documents_by_mean_score.parquet"

# Options:
# - "char": best for exact copy-paste and small edits
# - "word": good for word/phrase reuse
TFIDF_MODE: Literal["char", "word"] = "char"

# Change this to True only when rebuilding the TF-IDF source index.
BUILD_INDEX = True


# ============================================================
# LOAD CHUNKS
# ============================================================

def load_tfidf_chunks(
    path: Path,
    text_column: str = "lsa_esa_text",
) -> pd.DataFrame:
    df = pd.read_parquet(path)

    required_columns = {
        "chunk_id",
        "doc_id",
        "chunk_index",
        "start_char",
        "end_char",
        text_column,
    }

    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns in {path}: {missing}")

    df = df.copy()
    df[text_column] = df[text_column].fillna("").astype(str)
    df = df[df[text_column].str.strip() != ""].reset_index(drop=True)

    return df


# ============================================================
# VECTORIZER
# ============================================================

def make_tfidf_vectorizer(
    mode: Literal["char", "word"],
    max_features: int,
) -> TfidfVectorizer:
    if mode == "char":
        return TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(5, 8),
            max_features=max_features,
            lowercase=True,
            strip_accents="unicode",
            sublinear_tf=True,
            norm="l2",
        )

    if mode == "word":
        return TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 3),
            max_features=max_features,
            min_df=2,
            max_df=0.85,
            lowercase=True,
            strip_accents="unicode",
            sublinear_tf=True,
            norm="l2",
        )

    raise ValueError(f"Unsupported TF-IDF mode: {mode}")


# ============================================================
# BUILD TF-IDF INDEX
# ============================================================

def build_tfidf_index(
    source_chunks_path: Path,
    artifact_dir: Path,
    text_column: str = "lsa_esa_text",
    mode: Literal["char", "word"] = "char",
    max_features: int = 300_000,
    max_source_chunks: Optional[int] = None,
) -> None:
    """
    Build and save the TF-IDF source index.

    Creates:
    - tfidf_vectorizer.joblib
    - source_tfidf_vectors.npz
    - source_tfidf_metadata.parquet
    - tfidf_config.joblib
    """

    artifact_dir = Path(artifact_dir) / mode
    artifact_dir.mkdir(parents=True, exist_ok=True)

    source_df = load_tfidf_chunks(
        path=source_chunks_path,
        text_column=text_column,
    )

    if max_source_chunks is not None:
        source_df = source_df.head(max_source_chunks).reset_index(drop=True)

    source_texts = source_df[text_column].tolist()

    print(f"Loaded {len(source_df)} source chunks")
    print(f"Building TF-IDF index with mode: {mode}")
    print(f"Max features: {max_features}")

    vectorizer = make_tfidf_vectorizer(
        mode=mode,
        max_features=max_features,
    )

    print("Fitting TF-IDF...")
    source_tfidf = vectorizer.fit_transform(source_texts)

    # TfidfVectorizer already applies L2 norm with norm="l2".
    # This keeps cosine similarity explicit.
    source_tfidf = normalize(source_tfidf, norm="l2", axis=1)

    print(f"Source TF-IDF matrix shape: {source_tfidf.shape}")

    metadata_columns = [
        "chunk_id",
        "doc_id",
        "chunk_index",
        "start_char",
        "end_char",
    ]

    optional_columns = ["file_name", "relative_path", "part", "word_count"]
    metadata_columns += [col for col in optional_columns if col in source_df.columns]

    source_metadata = source_df[metadata_columns].copy()

    print("Saving TF-IDF artifacts...")

    joblib.dump(vectorizer, artifact_dir / "tfidf_vectorizer.joblib")
    sparse.save_npz(artifact_dir / "source_tfidf_vectors.npz", source_tfidf)

    source_metadata.to_parquet(
        artifact_dir / "source_tfidf_metadata.parquet",
        index=False,
    )

    config = {
        "mode": mode,
        "text_column": text_column,
        "max_features": max_features,
        "max_source_chunks": max_source_chunks,
        "source_chunks_path": str(source_chunks_path),
        "matrix_shape": source_tfidf.shape,
        "similarity": "cosine_similarity_via_l2_normalized_dot_product",
    }

    joblib.dump(config, artifact_dir / "tfidf_config.joblib")

    print(f"Saved TF-IDF artifacts to: {artifact_dir}")


# ============================================================
# LOAD TF-IDF INDEX
# ============================================================

def load_tfidf_index(
    artifact_dir: Path,
    mode: Literal["char", "word"] = "char",
):
    artifact_dir = Path(artifact_dir) / mode

    vectorizer_path = artifact_dir / "tfidf_vectorizer.joblib"
    vectors_path = artifact_dir / "source_tfidf_vectors.npz"
    metadata_path = artifact_dir / "source_tfidf_metadata.parquet"
    config_path = artifact_dir / "tfidf_config.joblib"

    for path in [vectorizer_path, vectors_path, metadata_path, config_path]:
        if not path.exists():
            raise FileNotFoundError(f"Missing TF-IDF artifact: {path}")

    print("Loading TF-IDF artifacts...")

    vectorizer = joblib.load(vectorizer_path)
    source_tfidf = sparse.load_npz(vectors_path)
    source_metadata = pd.read_parquet(metadata_path)
    config = joblib.load(config_path)

    return vectorizer, source_tfidf, source_metadata, config


# ============================================================
# QUERY ONE SUSPICIOUS DOCUMENT
# ============================================================

def retrieve_tfidf_candidates_for_suspicious_doc(
    suspicious_chunks_path: Path,
    artifact_dir: Path,
    suspicious_doc_id: str,
    output_path: Path,
    text_column: str = "lsa_esa_text",
    mode: Literal["char", "word"] = "char",
    top_k: int = 20,
    batch_size: int = 8,
    max_suspicious_chunks: Optional[int] = None,
) -> pd.DataFrame:
    """
    Query the TF-IDF source index using one selected suspicious document.

    Returns top-k source chunks for each suspicious chunk.
    """

    vectorizer, source_tfidf, source_metadata, config = load_tfidf_index(
        artifact_dir=artifact_dir,
        mode=mode,
    )

    suspicious_df = load_tfidf_chunks(
        path=suspicious_chunks_path,
        text_column=text_column,
    )

    suspicious_df = suspicious_df[
        suspicious_df["doc_id"] == suspicious_doc_id
    ].copy()

    if suspicious_df.empty:
        raise ValueError(f"No suspicious chunks found for doc_id: {suspicious_doc_id}")

    if max_suspicious_chunks is not None:
        suspicious_df = suspicious_df.head(max_suspicious_chunks).reset_index(drop=True)

    print(f"Selected suspicious document: {suspicious_doc_id}")
    print(f"Suspicious chunks to query: {len(suspicious_df)}")
    print(f"Searching top-{top_k} source chunks per suspicious chunk")
    print(f"TF-IDF mode: {mode}")

    results = []

    for start in range(0, len(suspicious_df), batch_size):
        end = min(start + batch_size, len(suspicious_df))
        batch_df = suspicious_df.iloc[start:end]

        batch_texts = batch_df[text_column].tolist()

        suspicious_tfidf = vectorizer.transform(batch_texts)
        suspicious_tfidf = normalize(suspicious_tfidf, norm="l2", axis=1)

        # Sparse cosine similarity.
        # Since both matrices are L2-normalized, dot product = cosine similarity.
        similarities = suspicious_tfidf @ source_tfidf.T

        for local_i, suspicious_row in enumerate(batch_df.itertuples(index=False)):
            sims_sparse = similarities.getrow(local_i)

            if sims_sparse.nnz == 0:
                continue

            candidate_indices = sims_sparse.indices
            candidate_scores = sims_sparse.data

            safe_top_k = min(top_k, len(candidate_scores))

            top_local_indices = np.argpartition(
                -candidate_scores,
                safe_top_k - 1,
            )[:safe_top_k]

            top_local_indices = top_local_indices[
                np.argsort(-candidate_scores[top_local_indices])
            ]

            for rank, local_idx in enumerate(top_local_indices, start=1):
                source_idx = int(candidate_indices[local_idx])
                score = float(candidate_scores[local_idx])

                source_row = source_metadata.iloc[source_idx]

                results.append({
                    "suspicious_chunk_id": suspicious_row.chunk_id,
                    "suspicious_doc_id": suspicious_row.doc_id,
                    "suspicious_chunk_index": suspicious_row.chunk_index,
                    "suspicious_start_char": suspicious_row.start_char,
                    "suspicious_end_char": suspicious_row.end_char,

                    "source_chunk_id": source_row["chunk_id"],
                    "source_doc_id": source_row["doc_id"],
                    "source_chunk_index": source_row["chunk_index"],
                    "source_start_char": source_row["start_char"],
                    "source_end_char": source_row["end_char"],

                    "TFIDF_score": score,
                    "TFIDF_rank": rank,
                    "TFIDF_mode": mode,
                })

        print(f"Processed suspicious chunks {start} to {end}")

    output_df = pd.DataFrame(results)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # Avoid overwriting char/word outputs by accident.
    output_path = output_path.with_name(
        output_path.stem + f"_{mode}" + output_path.suffix
    )

    output_df.to_parquet(output_path, index=False)

    print(f"Saved TF-IDF candidates to: {output_path}")

    return output_df


# ============================================================
# DOCUMENT-LEVEL MAX SCORE
# ============================================================

def get_top_source_documents_by_max_tfidf_score(
    candidates_df: pd.DataFrame,
    source_chunks_path: Path,
    suspicious_doc_id: str,
    top_n: int = 20,
    min_match_count: int = 1,
    output_path: Optional[Path] = None,
) -> pd.DataFrame:
    """
    Rank unique source documents by strongest TF-IDF chunk match.

    Best for copy-paste retrieval because plagiarism is often local.
    """

    required_columns = {
        "suspicious_doc_id",
        "suspicious_chunk_id",
        "source_doc_id",
        "source_chunk_id",
        "TFIDF_score",
    }

    missing = required_columns - set(candidates_df.columns)
    if missing:
        raise ValueError(f"Missing columns in candidates_df: {missing}")

    filtered_df = candidates_df[
        candidates_df["suspicious_doc_id"] == suspicious_doc_id
    ].copy()

    if filtered_df.empty:
        raise ValueError(f"No candidates found for suspicious_doc_id: {suspicious_doc_id}")

    grouped_df = (
        filtered_df
        .groupby("source_doc_id")
        .agg(
            mean_TFIDF_score=("TFIDF_score", "mean"),
            max_TFIDF_score=("TFIDF_score", "max"),
            min_TFIDF_score=("TFIDF_score", "min"),
            match_count=("TFIDF_score", "count"),
            unique_source_chunks=("source_chunk_id", "nunique"),
            unique_suspicious_chunks=("suspicious_chunk_id", "nunique"),
        )
        .reset_index()
    )

    grouped_df = grouped_df[grouped_df["match_count"] >= min_match_count].copy()

    grouped_df = (
        grouped_df
        .sort_values(
            ["max_TFIDF_score", "unique_suspicious_chunks", "match_count", "mean_TFIDF_score"],
            ascending=[False, False, False, False],
        )
        .head(top_n)
        .reset_index(drop=True)
    )

    grouped_df["source_doc_rank"] = range(1, len(grouped_df) + 1)

    grouped_df = grouped_df[
        [
            "source_doc_rank",
            "source_doc_id",
            "mean_TFIDF_score",
            "max_TFIDF_score",
            "min_TFIDF_score",
            "match_count",
            "unique_source_chunks",
            "unique_suspicious_chunks",
        ]
    ]

    source_meta_df = pd.read_parquet(
        source_chunks_path,
        columns=["doc_id", "relative_path"],
    ).drop_duplicates("doc_id")

    source_meta_df = source_meta_df.rename(columns={
        "doc_id": "source_doc_id",
        "relative_path": "source_relative_path",
    })

    grouped_df = grouped_df.merge(
        source_meta_df,
        on="source_doc_id",
        how="left",
    )

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)

        mode = candidates_df["TFIDF_mode"].iloc[0] if "TFIDF_mode" in candidates_df.columns else "unknown"
        output_path = output_path.with_name(
            output_path.stem + f"_{mode}" + output_path.suffix
        )

        grouped_df.to_parquet(output_path, index=False)
        print(f"Saved top source documents to: {output_path}")

    return grouped_df


# ============================================================
# DOCUMENT-LEVEL MEAN SCORE
# ============================================================

def get_top_source_documents_by_mean_tfidf_score(
    candidates_df: pd.DataFrame,
    source_chunks_path: Path,
    suspicious_doc_id: str,
    top_n: int = 20,
    min_match_count: int = 4,
    output_path: Optional[Path] = None,
) -> pd.DataFrame:
    """
    Rank unique source documents by mean TF-IDF score.
    """

    required_columns = {
        "suspicious_doc_id",
        "suspicious_chunk_id",
        "source_doc_id",
        "source_chunk_id",
        "TFIDF_score",
    }

    missing = required_columns - set(candidates_df.columns)
    if missing:
        raise ValueError(f"Missing columns in candidates_df: {missing}")

    filtered_df = candidates_df[
        candidates_df["suspicious_doc_id"] == suspicious_doc_id
    ].copy()

    if filtered_df.empty:
        raise ValueError(f"No candidates found for suspicious_doc_id: {suspicious_doc_id}")

    grouped_df = (
        filtered_df
        .groupby("source_doc_id")
        .agg(
            mean_TFIDF_score=("TFIDF_score", "mean"),
            max_TFIDF_score=("TFIDF_score", "max"),
            min_TFIDF_score=("TFIDF_score", "min"),
            match_count=("TFIDF_score", "count"),
            unique_source_chunks=("source_chunk_id", "nunique"),
            unique_suspicious_chunks=("suspicious_chunk_id", "nunique"),
        )
        .reset_index()
    )

    grouped_df = grouped_df[grouped_df["match_count"] >= min_match_count].copy()

    grouped_df = (
        grouped_df
        .sort_values(
            ["mean_TFIDF_score", "match_count", "max_TFIDF_score"],
            ascending=[False, False, False],
        )
        .head(top_n)
        .reset_index(drop=True)
    )

    grouped_df["source_doc_rank"] = range(1, len(grouped_df) + 1)

    grouped_df = grouped_df[
        [
            "source_doc_rank",
            "source_doc_id",
            "mean_TFIDF_score",
            "max_TFIDF_score",
            "min_TFIDF_score",
            "match_count",
            "unique_source_chunks",
            "unique_suspicious_chunks",
        ]
    ]

    source_meta_df = pd.read_parquet(
        source_chunks_path,
        columns=["doc_id", "relative_path"],
    ).drop_duplicates("doc_id")

    source_meta_df = source_meta_df.rename(columns={
        "doc_id": "source_doc_id",
        "relative_path": "source_relative_path",
    })

    grouped_df = grouped_df.merge(
        source_meta_df,
        on="source_doc_id",
        how="left",
    )

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)

        mode = candidates_df["TFIDF_mode"].iloc[0] if "TFIDF_mode" in candidates_df.columns else "unknown"
        output_path = output_path.with_name(
            output_path.stem + f"_{mode}" + output_path.suffix
        )

        grouped_df.to_parquet(output_path, index=False)
        print(f"Saved top source documents to: {output_path}")

    return grouped_df


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    BUILD_INDEX = True
    TFIDF_MODE = "char"

    SUSPICIOUS_DOC_ID = "part1__suspicious-document00007.txt"

    if BUILD_INDEX:
        build_tfidf_index(
            source_chunks_path=SOURCE_CHUNKS_PATH,
            artifact_dir=ARTIFACT_DIR,
            text_column="lsa_esa_text",
            mode=TFIDF_MODE,
            max_features=300_000,
            max_source_chunks=None,
        )

    candidates_df = retrieve_tfidf_candidates_for_suspicious_doc(
        suspicious_chunks_path=SUSPICIOUS_CHUNKS_PATH,
        artifact_dir=ARTIFACT_DIR,
        suspicious_doc_id=SUSPICIOUS_DOC_ID,
        output_path=OUTPUT_CANDIDATES_PATH,
        text_column="lsa_esa_text",
        mode=TFIDF_MODE,
        top_k=500,
        batch_size=8,
        max_suspicious_chunks=None,
    )

    top_sources_max_df = get_top_source_documents_by_max_tfidf_score(
        candidates_df=candidates_df,
        source_chunks_path=SOURCE_CANONICAL_CHUNKS_PATH,
        suspicious_doc_id=SUSPICIOUS_DOC_ID,
        top_n=50,
        min_match_count=1,
        output_path=OUTPUT_TOP_DOCS_MAX_PATH,
    )

    top_sources_mean_df = get_top_source_documents_by_mean_tfidf_score(
        candidates_df=candidates_df,
        source_chunks_path=SOURCE_CANONICAL_CHUNKS_PATH,
        suspicious_doc_id=SUSPICIOUS_DOC_ID,
        top_n=50,
        min_match_count=1,
        output_path=OUTPUT_TOP_DOCS_MEAN_PATH,
    )

    #print("\nTOP SOURCE DOCUMENTS BY MAX TF-IDF SCORE")
    #print(top_sources_max_df.to_string(index=False))

    #print("\nTOP SOURCE DOCUMENTS BY MEAN TF-IDF SCORE")
    #print(top_sources_mean_df.to_string(index=False))

top_sources_max_df

FileNotFoundError: [Errno 2] No such file or directory: '..\\..\\datasets\\processed\\PAN2011_300\\source_chunks_lsa_esa.parquet'